# Lab 6: HuggingFace for Computer Vision

In this lab, we will explore the HuggingFace ecosystem for Computer Vision tasks. We will cover zero-shot segmentation with SAM 3, visual embeddings with CLIP and DINOv3, and multimodal reasoning with Qwen3.5-0.8B.

## 1. Environment Setup & Memory Management

First, we install the necessary libraries and define a utility to keep our GPU memory clean.

In [ ]:
# Install relevant libraries if in Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q transformers accelerate bitsandbytes pillow torch torchvision

import torch
import gc
from PIL import Image
import requests
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

def flush_memory():
    """Utility to clear GPU memory between exercises to avoid OOM."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print("Memory flushed.")

## 2. HuggingFace Pipeline API

The `pipeline` API is the fastest way to use a model. Let's classify an image of a bus.

In [ ]:
from transformers import pipeline

# Exercise 1: Zero-shot Image Classification
classifier = pipeline("image-classification", model="google/vit-base-patch16-224")
url = "https://ultralytics.com/images/bus.jpg"
result = classifier(url)

for r in result:
    print(f"{r['label']}: {r['score']:.4f}")

del classifier
flush_memory()

## 3. SAM 3 - Segment Anything

SAM 3 introduces Promptable Concept Segmentation (PCS), allowing you to segment objects using text prompts.

In [ ]:
from transformers import Sam3Model, Sam3Processor

model_id = "facebook/sam3-h"
processor = Sam3Processor.from_pretrained(model_id)
model = Sam3Model.from_pretrained(model_id).to("cuda" if torch.cuda.is_available() else "cpu")

image = Image.open(requests.get("https://ultralytics.com/images/bus.jpg", stream=True).raw)

# Exercise 2: Segment by text
inputs = processor(images=image, text="yellow school bus", return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

results = processor.post_process_instance_segmentation(
    outputs, threshold=0.5, target_sizes=inputs.get("original_sizes").tolist()
)[0]

print(f"Detected {len(results['masks'])} masks for 'yellow school bus'")

del model, processor
flush_memory()

## 4. Vision Embeddings (CLIP & DINOv3)

Embeddings map images into a semantic space. CLIP connects text to images, while DINO focuses on pure visual features.

In [ ]:
from transformers import CLIPModel, CLIPProcessor, AutoModel, AutoImageProcessor

# Exercise 3: CLIP Zero-shot Classification
clip_id = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_id).to("cuda" if torch.cuda.is_available() else "cpu")
clip_proc = CLIPProcessor.from_pretrained(clip_id)

labels = ["a photo of a bus", "a photo of a city scene", "a photo of a desert"]
inputs = clip_proc(text=labels, images=image, return_tensors="pt", padding=True).to(clip_model.device)

with torch.no_grad():
    outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)

for label, prob in zip(labels, probs[0]):
    print(f"{label}: {prob.item():.4f}")

del clip_model, clip_proc
flush_memory()

### DINOv3 Similarity Search

Now let's use DINOv3 to compare two images.

In [ ]:
dino_id = "facebook/dinov3-vitb16-pretrain-lvd1689m"
dino_model = AutoModel.from_pretrained(dino_id).to("cuda" if torch.cuda.is_available() else "cpu")
dino_proc = AutoImageProcessor.from_pretrained(dino_id)

def get_embedding(img_url):
    img = Image.open(requests.get(img_url, stream=True).raw).convert("RGB")
    inputs = dino_proc(images=img, return_tensors="pt").to(dino_model.device)
    with torch.no_grad():
        out = dino_model(**inputs)
    return F.normalize(out.pooler_output, dim=-1)

# Exercise 4: Image Similarity
emb1 = get_embedding("https://ultralytics.com/images/bus.jpg")
emb2 = get_embedding("https://ultralytics.com/images/boats.jpg")

similarity = (emb1 @ emb2.T).item()
print(f"Similarity between Bus and Boats: {similarity:.4f}")

del dino_model, dino_proc
flush_memory()

## 5. Vision Language Models (Qwen3.5-0.8B)

The light-weight Qwen3.5-0.8B is an excelente VLM for on-device reasoning and OCR.

In [ ]:
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

vlm_id = "Qwen/Qwen3.5-0.8B"
vlm_model = Qwen2VLForConditionalGeneration.from_pretrained(
    vlm_id, torch_dtype="auto", device_map="auto"
)
vlm_proc = AutoProcessor.from_pretrained(vlm_id)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "https://ultralytics.com/images/bus.jpg"},
            {"type": "text", "text": "Describe what color the vehicles in this image are."},
        ],
    }
]

# Exercise 5: VQA
text = vlm_proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = vlm_proc.image_processor_fetcher(messages)
inputs = vlm_proc(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to(vlm_model.device)

generated_ids = vlm_model.generate(**inputs, max_new_tokens=128)
response = vlm_proc.batch_decode(generated_ids, skip_special_tokens=True)
print(response[0])

del vlm_model, vlm_proc
flush_memory()